In [ ]:
import csv
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib.pyplot import *
import pandas as pd
import numpy as np
from scipy import interpolate
from astropy.io import ascii
from matplotlib.font_manager import FontProperties
from matplotlib import font_manager




plt.style.use('seaborn-poster')
plt.rc('font', family='serif')
rcParams.update({'xtick.major.pad': '7.0'})
rcParams.update({'xtick.major.size': '7.5'})
rcParams.update({'xtick.major.width': '1.5'})
rcParams.update({'xtick.minor.pad': '7.0'})
rcParams.update({'xtick.minor.size': '3.5'})
rcParams.update({'xtick.minor.width': '1.0'})
rcParams.update({'ytick.major.pad': '7.0'})
rcParams.update({'ytick.major.size': '7.5'})
rcParams.update({'ytick.major.width': '1.5'})
rcParams.update({'ytick.minor.pad': '7.0'})
rcParams.update({'ytick.minor.size': '3.5'})
rcParams.update({'ytick.minor.width': '1.0'})
rcParams.update({'xtick.color': 'k'})
rcParams.update({'font.size': 12})

font_prop = font_manager.FontProperties(family = 'sans-serif', style = 'normal', size = 7)

: 

In [ ]:
# Functions to calculate Hubble inclination
def hubble_inclination(b_over_a, e=0.05):
    return np.sqrt((b_over_a**2 - e**2) / (1 - e**2))

# List initializations
b_over_a = []
twenty = []
nineteen = []
onefive = []
three = []
threefive = []

: 

In [ ]:
# Read new_SDSS_DR16_cosmos.csv file
data = ascii.read("/Users/ijimin/Downloads/Skyserver_SQL5_15_2024 12_07_27 PM.csv")
for i in range(len(data)):
    try:
        color_index = data['expMag_u'][i] - data['expMag_r'][i]
        condition = data['petroMag_r'][i] <= 21 and data['expMag_u'][i] < 22.15
        if condition and color_index < 2.3:
            b_over_a.append(hubble_inclination(data['expAB_r'][i]))
        if condition and color_index < 1.5:
            onefive.append(hubble_inclination(data['expAB_r'][i]))
        if condition and color_index < 3:
            three.append(hubble_inclination(data['expAB_r'][i]))
        if condition and color_index < 3.5:
            threefive.append(hubble_inclination(data['expAB_r'][i]))
    except:
        print('Error processing data from SDSS catalog')


# Read prmag_20.csv file
with open("/Users/ijimin/Downloads/prmag_20.csv") as file:
    tsv_file = csv.reader(file, delimiter=',')
    next(tsv_file, None)
    for line in tsv_file:
        if len(twenty) > 5000:
            break
        if line:
            line_data = line[0].split(',') if ',' in line[0] else line
            try:
                prmag = float(line_data[2])
                inclination = hubble_inclination(float(line_data[5]))
                if float(line_data[5]) > 0.05 and prmag <= 20.5:
                    twenty.append(inclination)
            except:
                print("Error processing line in prmag_20.csv")

# Read rpmag_19(1).csv file
with open("/Users/ijimin/Downloads/rpmag_19(1).csv") as file:
    tsv_file = csv.reader(file, delimiter=',')
    next(tsv_file, None)
    for line in tsv_file:
        if len(nineteen) > 5000:
            break
        if line:
            line_data = line[0].split(',') if ',' in line[0] else line
            try:
                prmag = float(line_data[2])
                inclination = hubble_inclination(float(line_data[5]))
                if float(line_data[5]) > 0.05:
                    nineteen.append(inclination)
            except:
                print("Error processing line in rpmag_19(1).csv")

bins_collected = []
means = []
stdevs = [[], []]
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
bins2 = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


: 

In [ ]:
#Read FRB host sample
df = pd.read_csv('/Users/ijimin/Desktop/FRBInclinationProject-main/my_data.csv')
ellips = df["ellipse"]

cosis = np.cos(np.radians(ellips))
cosis = cosis.sort_values()

#for 10000 times, take a sample of size of our FRB host sample, make a histogram, then add the histogram to bins collected
for i in range(0,len(bins2)):
  bins_collected.append([])
for i in range(0,10000):
  new_bins = np.histogram(random.sample(b_over_a,len(cosis)),bins=bins)

  for i in range(0,10):
    bins_collected[i].append(new_bins[0][i]/len(cosis))

#Get the means and 68% confidence region for each bin
for i in range(0,10):
  means.append(np.mean(bins_collected[i]))
  bins_collected[i].sort()
  stdevs[0].append(abs(means[i]-bins_collected[i][1600]))  
  stdevs[1].append(abs(bins_collected[i][8400]-means[i]))

b_over_a.append(1)

total_samples = []
means = []
stdevs = []
sigma_down = []
sigma_up = []
sdss_raw = []

b_over_a.sort()

#Get our index divided by len() as shown above
indices_normalized = [0]
for i in range(0,len(b_over_a)):
  indices_normalized.append(i/len(b_over_a))

b_over_a.insert(0,0)
#Interpolate between the points to create an actual function
f1 = interpolate.interp1d(b_over_a,indices_normalized)
f2 = interpolate.interp1d(indices_normalized,b_over_a)


: 

In [ ]:
#Create 10000 CDF functions of samples randomly taken from the SDSS sample. Works in the exact same way as described above.
for i in range(0,10000):

  new_sample = random.sample(b_over_a,len(cosis))

  indices_normalized_3 = []

  for i in range(1,len(cosis)+1):
    indices_normalized_3.append(i/len(cosis))

  new_sample.insert(0,0)
  new_sample.append(1)

  indices_normalized_3.append(1)
  indices_normalized_3.insert(0,0)

#The 10000 CDF function are put into a list of total_samples
  new_sample.sort()
  sdss_raw.append(new_sample)
  total_samples.append(interpolate.interp1d(new_sample,indices_normalized_3))

#At 100 locations from (0,1), sample each of the 1000 CDF functions at that location and get their mean and 68% confidence interval
for value in np.linspace(0,1,100):
  index_sample =[]
  for sample in total_samples:

    index_sample.append(sample(value))
  means.append(np.mean(index_sample))
  index_sample.sort()

#This is the 68% confidence interval for an ordered list of size 1000
  sigma_down.append(index_sample[1600])
  sigma_up.append(index_sample[8400])

: 

In [ ]:
data_i = []
list_cosi = pd.Series(cosis).tolist()
sum = 0
maxsum = 0
for x in list_cosi:
    maxsum += x
for x in list_cosi:
    sum += x
    data_i.append(sum/maxsum)

x = [0] + list_cosi + [1]
y = [0] + data_i + [1]

plt.figure(figsize=(8, 8))
plt.step(x, y, where='mid', color='red')
plt.legend(["FRB host sample ($m_r < 21$ mag AB)"], prop={'size': 16})
plt.ylabel("Cumulative distribution")
plt.xlabel("cos(i)")
# Set x-axis limits to range from 0 to 1
plt.xlim(0, 1)
plt.ylim(0, 1)
ax = plt.gca()
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
angle_degrees = [90, 78, 66, 53, 37, 0]
cos_values = np.cos(np.radians(angle_degrees))
ax2.set_xticks(cos_values)
ax2.set_xticklabels([f'{angle}°' for angle in angle_degrees])
ax2.set_xlabel('Inclination angle i (degrees)')
plt.show()

: 

In [ ]:
#make a cumulative distribution function for our SDSS sample
from scipy import interpolate

b_over_a.sort()

#Get our index divided by len() as shown above
indices_normalized = [0]

for i in range(0,len(b_over_a)):
  indices_normalized.append(i/len(b_over_a))


b_over_a.insert(0,0)

#Interpolate between the points to create an actual function
f1 = interpolate.interp1d(b_over_a,indices_normalized)

#Plot our function
plt.figure(figsize=(8, 8))
plt.plot(np.linspace(0,1,10000),f1(np.linspace(0,1,10000)))
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.title("CDF for SDSS")
plt.xlabel("cos(i)")
plt.ylabel("probability")

: 

In [ ]:
rmag = [19.95,
20.65,
14.97,
19.64,
16.44,
20.72,
19.77,
18.16,
19.14,
14.97,
20.34,
18.24,
17.73,
20.03,
19.65,
19.93,
20.33,
20.77,
17.41,
18.36,
15.39,
17.17,
19.47]
rmag.sort()
print(min(rmag),max(rmag))

: 

In [ ]:
cnt1 = 0
cnt2 = 0
cnt3 = 0
cnt4 = 0
cnt5 = 0
cnt6 = 0
cnt7 = 0
cnt8 = 0
cnt9 = 0
cnt10 = 0
cnt11 = 0
cnt12 = 0
cnt13 = 0
cnt14 = 0   
cnt15 = 0
cnt16 = 0
cnt17 = 0
cnt18 = 0
cnt19 = 0
cnt20 = 0


: 

In [ ]:
for value in rmag:
    if 14.97 <= value < 15.27:
        cnt1 += 1
    elif 15.27 <= value < 15.57:
        cnt2 += 1
    elif 15.57 <= value < 15.87:
        cnt3 += 1
    elif 15.87 <= value < 16.17:
        cnt4 += 1
    elif 16.17 <= value < 16.47:
        cnt5 += 1
    elif 16.47 <= value < 16.77:
        cnt6 += 1
    elif 16.77 <= value < 17.07:
        cnt7 += 1
    elif 17.07 <= value < 17.37:
        cnt8 += 1
    elif 17.37 <= value < 17.67:
        cnt9 += 1
    elif 17.67 <= value < 17.97:
        cnt10 += 1
    elif 17.97 <= value < 18.27:
        cnt11 += 1
    elif 18.27 <= value < 18.57:
        cnt12 += 1
    elif 18.57 <= value < 18.87:
        cnt13 += 1
    elif 18.87 <= value < 19.17:
        cnt14 += 1
    elif 19.17 <= value < 19.47:
        cnt15 += 1
    elif 19.47 <= value < 19.77:
        cnt16 += 1
    elif 19.77 <= value < 20.07:
        cnt17 += 1
    elif 20.07 <= value < 20.37:
        cnt18 += 1
    elif 20.37 <= value < 20.67:
        cnt19 += 1
    elif 20.67 <= value < 20.97:
        cnt20 += 1


        
cnts = [cnt1,cnt2,cnt3,cnt4,cnt5,cnt6,cnt7,cnt8,cnt9,cnt10,cnt11,cnt12,cnt13,cnt14,cnt15,cnt16,cnt17,cnt18,cnt19,cnt20]
print(cnts)

: 

In [ ]:
sdss = []
import csv

with open('/Users/ijimin/Desktop/new_SDSS_DR16_cosmos.csv') as file:
    tsv_file = csv.reader(file, delimiter=',')
    next(tsv_file, None)  # Skip the header line
    for line in tsv_file:
        if line:
            line_data = line[0].split(',') if ',' in line[0] else line     
            try:
                if float(line_data[1]) < 21 and float(line_data[11]) < 2.3:
                    sdss.append(line_data)
            except:
                print("Error")
check = []
for n in range(0, len(sdss)):
   check.append(sdss[n][4])
print(check)

: 

In [ ]:
u = 13.29
n = 0
bin1 = [] #13.29~13.79
bin2 = [] #13.79~14.29
bin3 = [] #14.29~14.79
bin4 = [] #14.79~15.29
bin5 = [] #15.29~15.79
bin6 = [] #15.79~16.29
bin7 = [] #16.29~16.79
bin8 = [] #16.79~17.29
bin9 = [] #17.29~17.79
bin10 = [] #17.79~18.29
bin11 = [] #18.29~18.79
bin12 = [] #18.79~19.29
bin13 = [] #19.29~19.79
bin14 = [] #19.79~20.29
bin15 = [] #20.29~20.79
bin16 = [] #20.79~21.29
bin17 = [] #21.29~21.79
bin18 = [] #21.79~22.29
bin19 = [] #22.29~22.79
bin20 = [] #22.79~23.29

bins = [bin1,bin2,bin3,bin4,bin5,bin6,bin7,bin8,bin9,bin10,bin11,bin12,bin13,bin14,bin15,bin16,bin17,bin18,bin19,bin20]



: 

In [ ]:
import csv
import numpy as np

n = 0

for line in sdss:
    mag = float(line[7])
    if 14.97 <= mag < 15.27:
        bin1.append(n)
    elif 15.27 <= mag < 15.57:
        bin2.append(n)
    elif 15.57 <= mag < 15.87:
        bin3.append(n)
    elif 15.87 <= mag < 16.17:
        bin4.append(n)
    elif 16.17 <= mag < 16.47:
        bin5.append(n)
    elif 16.47 <= mag < 16.77:
        bin6.append(n)
    elif 16.77 <= mag < 17.07:
        bin7.append(n)
    elif 17.07 <= mag < 17.37:
        bin8.append(n)
    elif 17.37 <= mag < 17.67:
        bin9.append(n)
    elif 17.67 <= mag < 17.97:
        bin10.append(n)
    elif 17.97 <= mag < 18.27:
        bin11.append(n)
    elif 18.27 <= mag < 18.57:
        bin12.append(n)
    elif 18.57 <= mag < 18.87:
        bin13.append(n)
    elif 18.87 <= mag < 19.17:
        bin14.append(n)
    elif 19.17 <= mag < 19.47:
        bin15.append(n)
    elif 19.47 <= mag < 19.77:
        bin16.append(n)
    elif 19.77 <= mag < 20.07:
        bin17.append(n)
    elif 20.07 <= mag < 20.37:
        bin18.append(n)
    elif 20.37 <= mag < 20.67:
        bin19.append(n)
    elif 20.67 <= mag < 20.97:
        bin20.append(n)
    n += 1


: 

In [ ]:
# Function to plot CDF with confidence intervals
def cdf_plot(data, color, label, ci=True, cosis_len=100):
    total_samples = []
    means = []
    sigma_down = []
    sigma_up = []
    
    for _ in range(1000):
        new_sample = random.sample(data, cosis_len)
        indices_normalized_3 = [i / cosis_len for i in range(1, cosis_len + 1)]
        new_sample.insert(0, 0)
        new_sample.append(1)
        indices_normalized_3.append(1)
        indices_normalized_3.insert(0, 0)
        new_sample.sort()
        total_samples.append(interpolate.interp1d(new_sample, indices_normalized_3))

    for value in np.linspace(0, 1, 100):
        index_sample = [sample(value) for sample in total_samples]
        means.append(np.mean(index_sample))
        index_sample.sort()

        if ci:
            sigma_down.append(index_sample[160])  # 16th percentile
            sigma_up.append(index_sample[840])    # 84th percentile

    plt.plot(np.linspace(0, 1, 100), means, color=color, label=label, linewidth=1)
    if ci:
        plt.fill_between(np.linspace(0, 1, 100), sigma_down, sigma_up, color=color, alpha=0.3)



: 

In [ ]:
def cdf_plot_rmag(data):
    print(data)
    total_samples = []
    means = []
    stdevs = []
    sigma_down = []
    sigma_up = []
    new_sample = []
    indexs = []
    random_choose = []
    
    for i in range(0,10000):
        new_sample = []
        indices_normalized_3 = []
        #Make new sample using the host rmag distribution
        for j in range(0,len(bins)):
            print(bins)
            indexs = random.sample(bins[j],cnts[j])
            for index in indexs:
                new_sample.append(float(sdss[index][4]))
        for i in range(1,len(cosis)+1):
            indices_normalized_3.append(i/len(cosis))
        new_sample.append(1)
        new_sample.insert(0,0)
        indices_normalized_3.append(1)
        indices_normalized_3.insert(0,0)
        new_sample.sort()
        total_samples.append(interpolate.interp1d(new_sample,indices_normalized_3))
    

    for value in np.linspace(0,1,100):
        sample_values = [sample(value) for sample in total_samples]
        mean = np.mean(sample_values)
        means.append(mean)
    
        sorted_samples = np.sort(sample_values)
        lower_bound = np.percentile(sorted_samples, 16)
        upper_bound = np.percentile(sorted_samples, 84)
    
        sigma_down.append(lower_bound)
        sigma_up.append(upper_bound)
    plt.plot(np.linspace(0,1,100),means,color='#377eb9')
    plt.fill_between(np.linspace(0,1,100),sigma_down, sigma_up,alpha = 0.5)

: 

In [ ]:
# Plot Combined FRB host sample with rPmag and u-r color thresholds
plt.figure(figsize=(6, 6))

# FRB host sample
x = [0, 0.17289330158437063, 0.36224813082482615, 0.4556550229487257, 0.5014876307532076, 
     0.5985261063679126, 0.6258535837434233, 0.6396877574727495, 0.6909344593104306, 
     0.7015702699958166, 0.7090078235351014, 0.7252436659386214, 0.7293778813452766, 
     0.7516105000429382, 0.7536197371667742, 0.7823447928818137, 0.8005793799984601, 
     0.8244675302234334, 0.836553999344333, 0.8837583627336427, 0.8922409988466021, 
     0.9063698713831745, 0.9306260493706966, 0.993124048144013, 1]
y = [0, 0.010627958576840851, 0.03289578557571118, 0.060905446244174564, 0.09173249227546486, 
     0.1285246097684138, 0.1669965800659247, 0.20631895361210248, 0.2487915233737468, 
     0.29191788917236616, 0.335501450305936, 0.38008304814158433, 0.4249187811559066, 
     0.47112117990226204, 0.5174470888640365, 0.5655387614468377, 0.6147513359660678, 
     0.6654323437255786, 0.7168563212356815, 0.7711820075693788, 0.826029131717062, 
     0.8817447746024875, 0.9389514738360852, 1.0, 1]
plt.step(x, y, where='mid', color='#e41a1c', label="FRB host sample", linewidth=1)

# Plot all CDFs together
cdf_plot(b_over_a, "#377eb8", "rPmag < 21 AB", True, len(b_over_a))
cdf_plot(twenty, "#4daf4a", "rPmag < 20 AB", False, len(twenty))
cdf_plot(nineteen, "#984ea3", "rPmag < 19 AB", False, len(nineteen))
cdf_plot(onefive, '#ff7f00', "u-r < 1.5", False, len(onefive))
cdf_plot(three, '#984ea3', "u-r < 3", False, len(three))
cdf_plot(threefive, '#8c564b', "u-r < 3.5", False, len(threefive))
cdf_plot(rmag, "17becf", "r-band host magnitude distribution", True, len(rmag))

plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend(prop=font_prop)
plt.ylabel("Cumulative distribution", fontsize=7)
plt.xlabel("cos(i)", fontsize=7)
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(1)

ax.set_xticks(np.arange(0.0, 1.2, 0.2))
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
angle_degrees = [90, 78, 66, 53, 37, 0]
cos_values = np.cos(np.radians(angle_degrees))
ax2.set_xticks(cos_values)
ax2.set_xticklabels([f'{angle}°' for angle in angle_degrees], fontproperties=font_prop)
ax2.set_xlabel('Inclination angle (degrees)', fontsize=7)

ax.set_xticklabels([f'{int(tick) if float(tick).is_integer() else tick:g}' for tick in ax.get_xticks()], fontproperties=font_prop)
ax.set_yticklabels([f'{int(tick) if float(tick).is_integer() else tick:g}' for tick in ax.get_yticks()], fontproperties=font_prop)

plt.savefig("combined_distributions.eps", bbox_inches="tight", dpi=300)
plt.show()

: 